In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# Load the dataset
delivery_time_path = os.path.join(path, 'Q1_data.csv')
df_delivery_time = pd.read_csv(delivery_time_path)

print(f"Dataset shape: {df_delivery_time.shape}")



In [ ]:
# Task 2: Write your code here:

df_delivery_time.head()


In [ ]:
# Task 3: Write your code here:

df_delivery_time.info()


In [ ]:
# Task 4: Write your code here:
df_delivery_time.describe()


In [ ]:
# Task 5: Write your code here:

# Target Distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_delivery_time['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task 1: Write your code here:

df_delivery_time.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_delivery_time)


In [ ]:
from pandas._libs import missing

missing_cols = ['Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs','Delivery_Time']

df_delivery_time[missing_cols] = df_delivery_time[missing_cols].fillna('unknown')
df_delivery_time

missing_values_check = df_delivery_time.isnull().sum()
missing_values_check

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_delivery_time)


In [ ]:
# Task 4: Write your code here:

categorical_cols = df_delivery_time.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))


# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type','Courier_Experience_yrs','Delivery_Time']
for col in categorical_cols:
    le = LabelEncoder()
    df_delivery_time[col] = le.fit_transform(df_delivery_time[col].astype(str))

df_delivery_time.head()

# Use One-Hot-Encoder for Bonus*******

In [ ]:
print("Missing values remaining:", df_delivery_time.isnull().sum().sum())

categorical_cols = df_delivery_time.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))


In [ ]:
# Task 5: Write your code here:
features = df_delivery_time.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

# Scale features - fit on train, transform both
scaler = StandardScaler()
df_delivery_time[features] = scaler.fit_transform(df_delivery_time[features])

df_delivery_time.head()
# ______________________________________

# # Define features (X) and target (y)
# feature_cols_template = ['feature1', 'feature2', 'engineered_feature'] # REPLACE WITH YOUR FEATURE COLUMNS
# target_col_template = 'Delivery_Time'

# X = df_delivery_time[feature_cols_template]
# y = df_delivery_time[target_col_template]

# # Split data (e.g., 80% train, 20% test)
# X_train_tpl, X_test_tpl, y_train_tpl, y_test_tpl = train_test_split(X, y, test_size=0.2, random_state=42)

# print(f"Train set shape: {X_train_tpl.shape}")
# print(f"Test set shape: {X_test_tpl.shape}")



In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df_delivery_time.drop("Delivery_Time",axis=1)
y = df_delivery_time['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# Task 2,3,4,5: Write your code here:

# Initialize K-Fold Cross-Validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in enumerate(kfold.split(X), start=1):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]


    # Initialize the model
    # Adjust hyperparameters like n_estimators, max_depth as needed
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


print("Model trained!")




In [ ]:
# Task 1: Write your code here:

coeffs = {}

coeffs['Lasso'] = model['LASSO Regression'].coef_
coeffs['Ridge'] = model['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: